# RMS-AV Multimodal Deepfake Detection: Cloud Execution & Training Pipeline

**Integrated Research Pipeline (Members 1, 2, and 3)**
- **Member 1:** Dataset preprocessing, frames/audio extraction, metadata manifest.
- **Member 2:** RMS-AV architecture (`AVDetector`), ResNet18 video encoder, 2D CNN audio encoder, and `ReliabilityGate`.
- **Member 3:** Cloud execution, multi-task training engine, leak-free manifests, checkpointing, and evaluation.

### Guarantees:
1. **Zero Identity Leakage:** Manifests are grouped by subject identity (`idXXXXX`).
2. **Missing Modality Robustness:** Tested under missing video (`has_video=False`) and missing audio (`has_audio=False`).
3. **Fair Baseline Comparison:** Identical splits and evaluation protocol across `audio`, `video`, `fusion`, and `rmsav`.

--- 
## 1. Mount Google Drive
Mounts Google Drive to load your dataset and persist checkpoints and logs across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive successfully mounted.")

--- 
## 2. Access / Clone Repository
Navigate into your project repository on Google Drive or clone it if running in an ephemeral session.

In [ ]:
import os

# Set your repository path on Drive or Colab filesystem
REPO_DIR = "/content/drive/MyDrive/DL--PROJECT"

if not os.path.exists(REPO_DIR):
    print(f"{REPO_DIR} not found on Drive. Using local clone placeholder...")
    # Replace with your actual GitHub URL if needed:
    # !git clone https://github.com/<YOUR_USER>/<YOUR_REPO>.git /content/DL--PROJECT
    # REPO_DIR = "/content/DL--PROJECT"

%cd {REPO_DIR}
!pwd

--- 
## 3. Install Compatible Pinned Dependencies

In [ ]:
!pip install -q -r requirements.txt
print("Dependencies installed successfully.")

--- 
## 4. Check Environment, PyTorch, CUDA, and GPU VRAM

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Active GPU: {device_name} ({vram_gb:.2f} GB VRAM)")
    !nvidia-smi
else:
    print("Running on CPU mode. Note: GPU is strongly recommended for video training.")

--- 
## 5. Configure Paths and Hyperparameters
Set the dataset location on Google Drive, output directories, and training options.

In [ ]:
import os

# =====================================================================
# DIRECTORY CONFIGURATION (Adjust DATA_DIR to your dataset location)
# =====================================================================
# If using Member 1's pilot data: DATA_DIR = "teammates/zip2"
# If using full FakeAVCeleb on Drive: DATA_DIR = "/content/drive/MyDrive/FakeAVCeleb_Dataset"
DATA_DIR = "/content/drive/MyDrive/FakeAVCeleb_Dataset"

SPLITS_DIR = "/content/drive/MyDrive/DL--PROJECT/splits"
CHECKPOINT_DIR = "/content/drive/MyDrive/DL--PROJECT/checkpoints"
RESULTS_DIR = "/content/drive/MyDrive/DL--PROJECT/results"

# Training Hyperparameters
BATCH_SIZE = 8       # Mini-batch size (4 or 8 on Colab T4)
EPOCHS = 10          # Epochs per model
LR = 1e-4            # Learning rate (Adam)
DIM = 256            # Projection embedding dimension
NUM_FRAMES = 5       # Frames per clip (T=5)
AUX_WEIGHT = 0.2     # Auxiliary loss weight for unimodal heads in RMS-AV
SEED = 42            # Reproducibility seed
USE_AMP = True       # Automatic Mixed Precision for VRAM efficiency

for d in [SPLITS_DIR, CHECKPOINT_DIR, RESULTS_DIR, f"{RESULTS_DIR}/eval"]:
    os.makedirs(d, exist_ok=True)

print("Paths and hyperparameters configured.")

--- 
## 6. Verify Dataset Files, Labels, & Generate Leakage-Safe Manifests
Inspects metadata, checks audio/frame availability, and generates persistent split manifests with zero identity leakage.

In [ ]:
from dataset import create_leakage_safe_splits

# Determine manifest location
base_manifest = os.path.join(DATA_DIR, "processed_manifest.csv")
if not os.path.exists(base_manifest):
    base_manifest = os.path.join(DATA_DIR, "meta_data.csv")

print(f"Using base manifest: {base_manifest}")
train_manifest, val_manifest, test_manifest = create_leakage_safe_splits(
    manifest_path=base_manifest,
    output_dir=SPLITS_DIR,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=SEED,
)
print(f"Manifests saved:\n - Train: {train_manifest}\n - Val: {val_manifest}\n - Test: {test_manifest}")

--- 
## 7. Run Synthetic Model & Pipeline Smoke Tests
Verifies models, reliability gate, data loader, training step, and evaluation before training on real data.

In [ ]:
!python test_models.py
!python test_dataset.py
!python test_train_pipeline.py
!python test_evaluate.py
print("\n>>> ALL INTEGRATION SMOKE TESTS PASSED! <<<")

--- 
## 8. Small Real-Data Sanity Run (1 Epoch)
Runs a 1-epoch sanity check with batch size 4 on real data to ensure zero CUDA OOM and proper I/O throughput.

In [ ]:
!python train.py \
    --mode rmsav \
    --manifest_dir {SPLITS_DIR} \
    --data_dir {DATA_DIR} \
    --batch_size 4 \
    --epochs 1 \
    --num_frames {NUM_FRAMES} \
    --save_dir {CHECKPOINT_DIR}/sanity \
    --log_dir {RESULTS_DIR}/sanity \
    --seed {SEED} \
    --amp

print("Sanity run completed successfully.")

--- 
## 9. Train Baseline Models (Audio-Only, Video-Only, Standard Fusion)
Trains all 3 baseline architectures on the exact same train/validation splits.

In [ ]:
for baseline_mode in ["audio", "video", "fusion"]:
    print(f"\n=======================================================")
    print(f"TRAINING BASELINE: {baseline_mode.upper()}")
    print(f"=======================================================")
    !python train.py \
        --mode {baseline_mode} \
        --manifest_dir {SPLITS_DIR} \
        --data_dir {DATA_DIR} \
        --batch_size {BATCH_SIZE} \
        --epochs {EPOCHS} \
        --lr {LR} \
        --num_frames {NUM_FRAMES} \
        --save_dir {CHECKPOINT_DIR} \
        --log_dir {RESULTS_DIR} \
        --seed {SEED} \
        --amp

--- 
## 10. Train Proposed Method: RMS-AV
Trains the full RMS-AV network with ReliabilityGate and unimodal multi-task supervision.

In [ ]:
print("\n=======================================================")
print("TRAINING PROPOSED METHOD: RMS-AV")
print("=======================================================")
!python train.py \
    --mode rmsav \
    --manifest_dir {SPLITS_DIR} \
    --data_dir {DATA_DIR} \
    --batch_size {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --lr {LR} \
    --aux_weight {AUX_WEIGHT} \
    --num_frames {NUM_FRAMES} \
    --save_dir {CHECKPOINT_DIR} \
    --log_dir {RESULTS_DIR} \
    --seed {SEED} \
    --amp

--- 
## 11. Final Evaluation & Stress Testing on Holdout Test Set
Evaluates all 4 models on `test_manifest.csv` under standard, missing video, and missing audio conditions.

In [ ]:
import glob
import json
import pandas as pd

MODES = ["audio", "video", "fusion", "rmsav"]
STRESS_CONDITIONS = ["none", "missing_video", "missing_audio"]

for m in MODES:
    ckpt = f"{CHECKPOINT_DIR}/{m}_best.pth"
    if not os.path.exists(ckpt):
        print(f"Checkpoint {ckpt} not found. Skipping.")
        continue
    for cond in STRESS_CONDITIONS:
        if m == "video" and cond == "missing_video":
            continue
        if m == "audio" and cond == "missing_audio":
            continue
        print(f"Evaluating {m} [Stress: {cond}]...")
        !python evaluate.py \
            --checkpoint {ckpt} \
            --test_manifest {SPLITS_DIR}/test_manifest.csv \
            --data_dir {DATA_DIR} \
            --batch_size {BATCH_SIZE} \
            --stress_test {cond} \
            --output_dir {RESULTS_DIR}/eval

# Summary Comparison Table
eval_files = glob.glob(f"{RESULTS_DIR}/eval/*_metrics.json")
rows = []
for f in eval_files:
    base = os.path.basename(f).replace("_metrics.json", "")
    with open(f) as jf:
        data = json.load(jf)
    data["experiment"] = base
    rows.append(data)

if rows:
    summary_df = pd.DataFrame(rows)[
        ["experiment", "accuracy", "balanced_accuracy", "f1_score", "auc", "fpr", "fnr", "ece"]
    ]
    print("\n=======================================================")
    print("FINAL RESEARCH COMPARISON SUMMARY (TEST SET)")
    print("=======================================================")
    print(summary_df.to_string(index=False))
    summary_df.to_csv(f"{RESULTS_DIR}/final_model_comparison.csv", index=False)
    print(f"Saved final comparison table to: {RESULTS_DIR}/final_model_comparison.csv")
else:
    print("No evaluation files found.")